# Reproduce the Fire VASE Pipeline

This notebook is the runnable companion to the website vignette. It walks through the manuscript-support workflow: environment checks, data-lake verification, optional data-lake rebuild commands, manuscript analysis products, manuscript figures, and final reproducibility checks.

The notebook assumes it is run from the root of the `fire_vase` repository. The quickest route is to download or sync the shared CyVerse data lake first, then run the verification and figure cells.

## 0. Configure the Run

Set paths and choose which parts of the pipeline to run. Full data-lake rebuilds are intentionally off by default because they require large FIRED and gridMET source caches and can take substantial time.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
DATA_LAKE = REPO_ROOT / "data_lake" / "fire-vase-data-lake-v0.1"
REPRO_REPORT = REPO_ROOT / "analysis" / "reproducibility_check_latest.json"

# Use the shared data lake for most collaborator workflows.
VERIFY_DATA_LAKE = True
RUN_ANALYSIS_PRODUCTS = False
RUN_FIGURES = True
RUN_FULL_SOURCE_REBUILD = False
REFRESH_DATA_LAKE_PACKAGE = False
FORCE_VALIDATION_TABLES = False

print(f"Repository: {REPO_ROOT}")
print(f"Python: {sys.executable}")
print(f"Data lake: {DATA_LAKE}")

In [ ]:
def run_command(args, *, active=True, cwd=REPO_ROOT):
    """Run a repository command, or print it when the step is disabled."""
    printable = " ".join(str(part) for part in args)
    if not active:
        print(f"Skipped: {printable}")
        return None
    print(f"Running: {printable}")
    return subprocess.run(args, cwd=cwd, check=True)


def require_repo_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Expected file or folder does not exist: {path}")
    return path


require_repo_file(REPO_ROOT / "pyproject.toml")
require_repo_file(REPO_ROOT / "scripts" / "check_reproducibility.py")
require_repo_file(REPO_ROOT / "manuscript_figures" / "00_run_all.py")

## 1. Set Up the Environment

From a terminal, clone the repository and install the locked environment before opening this notebook:

```bash
git clone https://github.com/CU-ESIIL/fire_vase.git
cd fire_vase
uv sync
```

If this notebook is already running in that environment, the next cell should complete without error.

In [ ]:
run_command(["uv", "run", "python", "--version"])

## 2. Choose a Data-Lake Starting Point

### Option A: Use the shared data lake

Download or sync the shared data lake from CyVerse:

https://de.cyverse.org/data/ds/iplant/home/shared/esiil/Fire_Vase?type=folder&resourceId=ce3e72e4-95d1-11f1-852a-90e2ba675364

Place it at:

```text
data_lake/fire-vase-data-lake-v0.1
```

Then run the verification cell below.

In [ ]:
if VERIFY_DATA_LAKE:
    require_repo_file(DATA_LAKE)

run_command(
    [
        "uv", "run", "python", "scripts/check_reproducibility.py",
        "--json-output", str(REPRO_REPORT),
    ],
    active=VERIFY_DATA_LAKE,
)

In [ ]:
if REPRO_REPORT.exists():
    report = json.loads(REPRO_REPORT.read_text())
    print(json.dumps({
        "data_lake": report.get("data_lake", {}).get("status"),
        "derived_stats": report.get("derived_stats", {}).get("status"),
        "figures_pixel": report.get("figures", {}).get("pixel_status"),
    }, indent=2))
else:
    print(f"No reproducibility report found at {REPRO_REPORT}")

### Option B: Rebuild the lake from source caches

Use this route only when the source FIRED and gridMET caches are available locally and you need to audit or recreate the manuscript-scale lakehouse. Turn on `RUN_FULL_SOURCE_REBUILD` in the configuration cell before running this section.

In [ ]:
run_command(
    ["uv", "run", "python", "scripts/cache_gridmet_years.py", "--preset", "comprehensive", "--keep-going"],
    active=RUN_FULL_SOURCE_REBUILD,
)

run_command(
    [
        "uv", "run", "python", "scripts/fire_vase_lakehouse_pilot.py",
        "--config", "config/fire_vase_pipeline.yml",
        "--output-root", "scratch/fire_vase_run_full",
        "--full-population",
    ],
    active=RUN_FULL_SOURCE_REBUILD,
)

run_command(
    [
        "uv", "run", "python", "scripts/fire_vase_build_climate_tables.py",
        "--include-optional-variables",
        "--table-root", "scratch/fire_vase_run_full/tables",
    ],
    active=RUN_FULL_SOURCE_REBUILD,
)

run_command(
    ["uv", "run", "python", "scripts/fire_vase_build_perimeter_climate_tables.py", "--include-optional-variables"],
    active=RUN_FULL_SOURCE_REBUILD,
)

run_command(
    [
        "uv", "run", "python", "scripts/fire_vase_developmental_morphology_analysis.py",
        "--table-root", "scratch/fire_vase_run_full/tables",
        "--data-output-dir", "scratch/fire_vase_developmental_morphology",
    ],
    active=RUN_FULL_SOURCE_REBUILD,
)

## 3. Regenerate Manuscript Analysis Products

This refreshes derived tables, summaries, and support products used by the manuscript. It is disabled by default for quick figure-only reproduction from the shared data lake.

In [ ]:
run_command(
    ["uv", "run", "python", "scripts/fire_vase_climate_revision.py"],
    active=RUN_ANALYSIS_PRODUCTS,
)

run_command(
    ["uv", "run", "python", "scripts/fire_vase_manuscript_claim_audit.py"],
    active=RUN_ANALYSIS_PRODUCTS,
)

## 4. Regenerate Manuscript Figures

The numbered figure scripts live in `manuscript_figures/`. The wrapper below runs the full figure set against the configured data lake. Set `FORCE_VALIDATION_TABLES = True` to recompute validation tables instead of using cached data-lake products.

In [ ]:
figure_cmd = [
    "uv", "run", "python", "manuscript_figures/00_run_all.py",
    "--data-lake", str(DATA_LAKE),
]
if FORCE_VALIDATION_TABLES:
    figure_cmd.append("--force-validation")

if RUN_FIGURES:
    require_repo_file(DATA_LAKE)

run_command(figure_cmd, active=RUN_FIGURES)

In [ ]:
for path in sorted((REPO_ROOT / "manuscript_figures").glob("Figure_*.png")):
    print(path.relative_to(REPO_ROOT))
for path in sorted((REPO_ROOT / "manuscript_figures").glob("Supplementary_Figure_*.png")):
    print(path.relative_to(REPO_ROOT))

## 5. Check the Whole Pipeline

Run the reproducibility checker after regenerating products. Use `--skip-data-lake` when the downloaded data lake has already been verified and you only want to check regenerated figures and derived statistics.

In [ ]:
run_command(
    [
        "uv", "run", "python", "scripts/check_reproducibility.py",
        "--json-output", str(REPRO_REPORT),
    ],
    active=VERIFY_DATA_LAKE,
)

run_command(
    [
        "uv", "run", "python", "scripts/check_reproducibility.py",
        "--skip-data-lake",
        "--json-output", str(REPRO_REPORT),
    ],
    active=not VERIFY_DATA_LAKE,
)

In [ ]:
if REPRO_REPORT.exists():
    report = json.loads(REPRO_REPORT.read_text())
    print(json.dumps(report, indent=2)[:4000])
else:
    print("Run the reproducibility checker to create the report.")

## 6. Refresh the Shareable Data-Lake Package

After intentional changes to scripts, figures, manuscripts, or derived tables, refresh the package manifest and checksums. Use `--mode manifest` for a metadata refresh, or `--mode copy` for an upload-ready materialized package. This is disabled by default.

In [ ]:
run_command(
    ["uv", "run", "python", "scripts/prepare_data_lake.py", "--mode", "manifest", "--checksum"],
    active=REFRESH_DATA_LAKE_PACKAGE,
)

## Code Map

- Data release config: `config/data_release.yml`
- Data-lake packager: `scripts/prepare_data_lake.py`
- Reproducibility checker: `scripts/check_reproducibility.py`
- Lakehouse builder: `scripts/fire_vase_lakehouse_pilot.py`
- Climate table builder: `scripts/fire_vase_build_climate_tables.py`
- Perimeter climate builder: `scripts/fire_vase_build_perimeter_climate_tables.py`
- Developmental morphology analysis: `scripts/fire_vase_developmental_morphology_analysis.py`
- Manuscript figure wrappers: `manuscript_figures/`